# Part 2 (실습) — 프롬프트 템플릿으로 질문 찍어내기

> **이 노트북의 목표**
> 빈칸이 뚫린 틀(템플릿)을 만들어 변수만 갈아 끼워 질문을 찍어냄. `PromptTemplate`, `ChatPromptTemplate`, `partial`, few-shot까지 직접 돌려봄.
>
> **사용 모델**: `gemini-3-flash` · **선행**: Part 0~1

## 0. 준비

In [ ]:
%pip install -q -U langchain langchain-google-genai

In [ ]:
import os
from getpass import getpass
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("Gemini API 키: ")
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-3-flash", temperature=0.7)
print("준비 완료")

## 1. 직접 조립의 한계부터 느껴보기

### 직접 조립 — 입력이 바뀔 때마다 문자열을 다시 만듦

In [ ]:
products = ["베이지 니트", "린넨 셔츠", "울 코트"]
for p in products:
    prompt = "다음 상품의 홍보 문구를 한 줄로 써줘: " + p   # 매번 + 조립
    print(p, "→", llm.invoke(prompt).content.strip())

문장 형식을 바꾸려면 위 `+` 부분을 전부 손봐야 함. 이걸 템플릿으로 바꿔봄.

## 2. PromptTemplate — 단일 문자열 템플릿

### 문법
- `PromptTemplate.from_template("... {변수} ...")` : 템플릿 생성, `{변수}` 자동 인식
- `.invoke({"변수": 값})` : 빈칸 채우기 → 완성된 프롬프트 반환 (`.text`에 문자열)

In [ ]:
from langchain_core.prompts import PromptTemplate

template = PromptTemplate.from_template(
    "다음 상품의 홍보 문구를 {tone} 톤으로 한 줄 써줘: {product}"
)
print("이 템플릿의 빈칸:", template.input_variables)

# 빈칸을 채워 완성된 프롬프트만 먼저 확인
filled = template.invoke({"product": "베이지 니트", "tone": "감성적인"})
print("완성된 프롬프트:", filled.text)

### 템플릿 + 모델을 이어서 호출
완성된 프롬프트를 그대로 모델에 넘김. (Part 3에서 이 두 줄이 파이프 `|` 한 줄이 됨)

In [ ]:
for p in products:
    prompt = template.invoke({"product": p, "tone": "감성적인"})
    print(p, "→", llm.invoke(prompt).content.strip())

## 3. ChatPromptTemplate — 메시지 템플릿

### 문법
- `ChatPromptTemplate.from_messages([("system","..."), ("human","...")])`
- 각 튜플 = (역할, 템플릿). 역할은 `"system"`, `"human"`, `"ai"`
- `.invoke({...})` → SystemMessage / HumanMessage 리스트 반환 (ChatModel에 그대로 투입 가능)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

chat_tmpl = ChatPromptTemplate.from_messages([
    ("system", "너는 리리마켓의 {tone} 쇼핑 도우미야. 답은 2문장 이내로 해."),
    ("human", "{product}에 어울리는 코디를 추천해줘."),
])

# 빈칸을 채우면 메시지 리스트가 만들어짐
msgs = chat_tmpl.invoke({"tone": "친절한", "product": "베이지 니트"})
for m in msgs.messages:
    print(f"[{type(m).__name__}] {m.content}")

In [ ]:
# 그대로 모델에 호출
res = llm.invoke(chat_tmpl.invoke({"tone": "친절한", "product": "베이지 니트"}))
print(res.content)

### tone만 바꿔 분위기 비교

In [ ]:
for tone in ["다정한", "시크하고 도시적인"]:
    res = llm.invoke(chat_tmpl.invoke({"tone": tone, "product": "트렌치 코트"}))
    print(f"[{tone}] {res.content}")
    print("-"*50)

## 4. partial — 일부 변수 미리 고정하기

### 문법
- `template.partial(변수=값)` : 일부 변수를 미리 채운 새 템플릿 반환
- 남은 변수만 나중에 `.invoke()`로 채움

브랜드명처럼 거의 안 바뀌는 값을 고정해 호출부를 깔끔하게 만듦.

In [ ]:
base = PromptTemplate.from_template("{brand}의 {product} 홍보 문구를 한 줄로 써줘")
print("원래 빈칸:", base.input_variables)

# brand를 '리리마켓'으로 고정
lily = base.partial(brand="리리마켓")
print("partial 후 남은 빈칸:", lily.input_variables)

print(lily.invoke({"product": "캐시미어 머플러"}).text)

> 📌 **함수도 partial 가능**: 현재 날짜처럼 매번 달라져야 하는 값은 고정 문자열 대신 함수를 걸어 호출 때마다 새로 계산되게 함.

In [ ]:
from datetime import datetime
dated = PromptTemplate.from_template("오늘은 {today}. {product} 오늘의 추천 문구를 써줘")
dated = dated.partial(today=lambda: datetime.now().strftime("%Y-%m-%d"))
print(dated.invoke({"product": "린넨 원피스"}).text)

## 5. Few-shot — 예시로 형식을 가르치기

### 개념
지시만 주는 zero-shot 대신, **잘 된 예시 몇 개**를 보여주면 형식·스타일을 훨씬 잘 따름.

### 문법 (FewShotPromptTemplate)
- `example_prompt` : 예시 하나를 어떻게 표현할지의 틀
- `examples` : 예시 데이터(딕셔너리 리스트)
- `suffix` : 예시 뒤에 붙는, 모델이 채울 새 질문 부분

In [ ]:
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate

example_prompt = PromptTemplate.from_template("상품: {product}\n문구: {copy}")

examples = [
    {"product": "베이지 니트", "copy": "포근함을 입다."},
    {"product": "린넨 셔츠", "copy": "여름을 가볍게."},
    {"product": "가죽 부츠", "copy": "걸음마다 깊어지는 멋."},
]

few_shot = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    suffix="상품: {product}\n문구:",
    input_variables=["product"],
)

# 완성된 프롬프트 먼저 확인 — 예시들이 어떻게 조립되는지 눈으로 봄
print(few_shot.invoke({"product": "울 코트"}).text)

In [ ]:
# 모델 호출 — 예시의 짧고 감성적인 형식을 따라 함
prompt = few_shot.invoke({"product": "울 코트"})
print("\n→ 모델 답:", llm.invoke(prompt).content.strip())

### zero-shot과 비교
예시 없이 같은 요청을 하면 형식이 들쭉날쭉해짐.

In [ ]:
zero = llm.invoke("울 코트의 홍보 문구를 써줘")
print("zero-shot:", zero.content.strip())
print()
print("few-shot :", llm.invoke(few_shot.invoke({"product": "울 코트"})).content.strip())

## ✏️ 미니 실습

**과제**: `ChatPromptTemplate`으로 "데이터 분석 용어를 중학생 눈높이로 한 문장 설명"하는 템플릿을 만들기.
- system: "너는 모든 걸 일상 비유로 설명하는 데이터 분석 강사야."
- human: "{term}을(를) 한 문장으로 설명해줘."
- `term="오버피팅"`으로 호출

In [ ]:
# TODO: 직접 작성
# tmpl = ChatPromptTemplate.from_messages([...])
# print(llm.invoke(tmpl.invoke({"term": "오버피팅"})).content)

<details><summary>👉 정답</summary>

```python
tmpl = ChatPromptTemplate.from_messages([
    ("system", "너는 모든 걸 일상 비유로 설명하는 데이터 분석 강사야."),
    ("human", "{term}을(를) 한 문장으로 설명해줘."),
])
print(llm.invoke(tmpl.invoke({"term": "오버피팅"})).content)
```
</details>

## 정리

| 절 | 한 일 | 핵심 |
|---|---|---|
| 2 | PromptTemplate | `from_template`, `{변수}`, `.invoke` |
| 3 | ChatPromptTemplate | `("system","...")` 튜플 → 메시지 리스트 |
| 4 | partial | 일부 변수(값·함수) 미리 고정 |
| 5 | few-shot | 예시로 출력 형식 학습 |

### 3줄 요약
1. 템플릿은 빈칸 양식지 — 변수만 갈아 끼워 질문을 찍어냄.
2. 채팅 모델엔 `ChatPromptTemplate`, 일부 고정은 `partial`.
3. few-shot은 예시 2~5개로 형식·스타일을 통제함.

### ▶ 다음 (Part 3)
드디어 `프롬프트 | 모델`을 파이프 하나로 잇는 **LCEL**을 배움. 초급의 정점임.